In [ ]:
import pandas as pd
from pathlib import Path
import pandas as pd


# 1. Load the processed data
PROCESSED_DIR = Path('../Data/processed')
df_events = pd.read_csv(PROCESSED_DIR / 'events/master_events.csv')
df_clubs = pd.read_csv(PROCESSED_DIR / 'clubs/master_clubs.csv')
df_matches = pd.read_csv(PROCESSED_DIR / 'events/master_match_results.csv')


In [ ]:

def search_clubs(query, match_df=df_clubs):
    """
    Performs a case-insensitive search for clubs matching the query.
    Returns a unique list of club names.
    """
    # 1. Extract unique club names from the data
    unique_clubs = pd.Series(match_df['Name'].unique())
    
    # 2. Filter using case-insensitive partial matching
    matches = unique_clubs[unique_clubs.str.contains(query, case=False, na=False)]
    
    # 3. Present results
    if matches.empty:
        print(f"No clubs found matching: '{query}'")
        return []
    else:
        print(f"Found {len(matches)} potential matches for '{query}':")
        return matches.tolist()

# --- Now you can run the search ---
results = search_clubs("Ocean")

if results:
    # Example: Picking the first result for further analysis
    my_target_club = results[0]
    print(f"\nTarget Club Selected: {my_target_club}")


In [ ]:
# 1. Search to find the exact spelling
results = search_clubs("Ocean")

# 2. Select the correct one (e.g., the first result)
my_target_club = results[0] 

# 3. Run your analysis using that exact name
club_matches = df_clubs[df_clubs['Name'] == my_target_club]
# ... and so on



In [ ]:
from thefuzz import process

def fuzzy_find_clubs(query, limit=5, match_df=df_clubs):
    """
    Uses Levenshtein Distance to find the closest matches even with typos.
    """
    unique_names = match_df['Name'].unique().tolist()
    
    # Extract the top 'limit' matches
    results = process.extract(query, unique_names, limit=limit)
    
    # Format: (Name, Confidence Score)
    print(f"Top matches for '{query}':")
    for name, score in results:
        print(f"[{score}% match] - {name}")

# --- Example Usage ---
# Even if you misspell it, it will find the closest name
fuzzy_find_clubs("MVP") 


In [ ]:
def get_teams_in_club(club_name, match_df=df_matches):
    """
    Finds all unique team names in the match data that belong to the specified club.
    """
    # 1. Get all unique names from both Team A and Team B columns
    all_teams = pd.concat([match_df['Team_A_Name'], match_df['Team_B_Name']]).unique()
    all_teams_series = pd.Series(all_teams).dropna()
    
    # 2. Filter teams that contain the club name
    # We use case-insensitive matching
    club_teams = all_teams_series[all_teams_series.str.contains(club_name, case=False)]
    
    if club_teams.empty:
        print(f"No teams found containing the name '{club_name}'.")
        return []
    else:
        print(f"Found {len(club_teams)} teams for club '{club_name}':")
        return sorted(club_teams.tolist())

# --- Example Usage ---
# Use the exact club name found from your 'search_clubs' routine
my_teams = get_teams_in_club("MVP")
print(my_teams)
